# Тестирование сервиса в ручную

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 500) 

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env").resolve()
load_dotenv(env_path)

HF_TOKEN = os.getenv("HF_TOKEN")

print(env_path)
print(HF_TOKEN is not None)

D:\Projects\speech_to_text_service\.env
True


In [3]:
import sys
import os

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)

from app.models.transcriber import FastWhisperTranscriber
from app.settings import settings

In [4]:
import json
from pathlib import Path

RESULT_JSON_FILE = settings.AUDIO_DIR / "debug_transcription_with_diarization.json"


In [5]:
import json

with RESULT_JSON_FILE.open("r", encoding="utf-8") as f:
    result = json.load(f)

text_segments = result.get("segments") or []
diarization_segments = (result.get("diarization") or {}).get("segments") or []

print("Result loaded:", RESULT_JSON_FILE)
print("segments:", len(text_segments))
print("diarization segments:", len(diarization_segments))

print("\n=== TEXT SEGMENTS ===")
for segment in text_segments:
    start = segment.get("start")
    end = segment.get("end")
    text = segment.get("text", "")

    print(f"[{start:.2f} - {end:.2f}] {text}")

print("\n=== DIARIZATION SEGMENTS ===")
for segment in diarization_segments:
    speaker = segment.get("speaker")
    start = segment.get("start")
    end = segment.get("end")
    duration = segment.get("duration")

    if duration is None and start is not None and end is not None:
        duration = end - start

    print(f"{speaker}: {start:.2f} -> {end:.2f} ({duration:.2f}s)")

Result loaded: D:\Projects\speech_to_text_service\data\audio\debug_transcription_with_diarization.json
segments: 140
diarization segments: 155

=== TEXT SEGMENTS ===
[19.03 - 30.10] Ольга Санна, слушайте меня. Когда мы его поймаем, ни о какой Женевской конвенции я не хочу слышать. Просто отдадите его мне на растерзание.
[31.06 - 32.06] У вас что, арбалет?
[32.69 - 36.59] Да, еще светошумовая граната, сюрикены и песок, что бросить ему в глаза.
[37.80 - 52.95] Денис, пусть сюрикены, вы что, мы датчика ищем или Рэмбо? Задолбали эти закладчики. Да посмотрите, как они нам двор весь перерыли, у нас двор весь в ямках, как лицо студента.
[52.95 - 62.33] Ну все, я обход закончил, вроде все спокойно.
[63.05 - 65.23] Спокойно? А что так запыхались?
[65.45 - 67.77] Да там собаки слиплись, я разлеплял.
[71.21 - 74.53] Блин, поймали бы уже этого закладчика, надоел он.
[75.15 - 84.13] Постоянно наркоманы эти приходят к нам во двор, колют тут свои марихуаны, пьют кокаины свои, надоели уже.
[84.83 - 89

In [6]:
from app.utils.exporters.common import build_speaker_blocks

speaker_blocks = build_speaker_blocks(result)

print("\n=== TEXT + SPEAKERS ===")
for block in speaker_blocks:
    speaker = block.get("speaker") or "Без спикера"
    parts = block.get("parts") or []
    if not parts:
        continue

    start = parts[0].get("start")
    end = parts[-1].get("end")
    text = " ".join(part["text"] for part in parts)

    print(f"\n{speaker} [{start:.2f} - {end:.2f}]")
    print(text)


=== TEXT + SPEAKERS ===

Спикер 01 [19.03 - 30.10]
Ольга Санна, слушайте меня. Когда мы его поймаем, ни о какой Женевской конвенции я не хочу слышать. Просто отдадите его мне на растерзание.

Спикер 02 [31.06 - 31.96]
У вас что, арбалет?

Спикер 01 [32.69 - 36.59]
Да, еще светошумовая граната, сюрикены и песок, что бросить ему в глаза.

Спикер 02 [37.80 - 52.95]
Денис, пусть сюрикены, вы что, мы датчика ищем или Рэмбо? Задолбали эти закладчики. Да посмотрите, как они нам двор весь перерыли, у нас двор весь в ямках, как лицо студента.

Спикер 03 [52.95 - 89.33]
Ну все, я обход закончил, вроде все спокойно. Спокойно? А что так запыхались? Да там собаки слиплись, я разлеплял. Блин, поймали бы уже этого закладчика, надоел он. Постоянно наркоманы эти приходят к нам во двор, колют тут свои марихуаны, пьют кокаины свои, надоели уже. Во дворе столько наркоты примагничено, что магнитное поле образовалось.

Спикер 01 [89.33 - 121.35]
Вон счетчики все в квартире повылетали. Согласен, с алкашами 